# WithU Talk — Phase 2: Context / Sentiment Module

Module **B** of the ensemble. Where Phase 1 judges a single message, this judges a **window of recent
messages** — so it can tell an escalating pile-on from isolated banter, which is the whole point of
"context."

**Design (buildable with the data you actually have):** a window-level binary classifier on
**KLUE-RoBERTa**. A window is **positive** if it contains verbal-harm (your `cb_binary=1` messages in
context), **negative** otherwise. Negatives come from three places so the model can't cheat:
- **AI-Hub SNS/topic multi-turn** dialogues → ordinary chat (the payoff of that download);
- **WithU non-CB events** (WEX/WRV) → *in-domain* non-bullying (stops the model keying on register);
- calm stretches inside CB events.

> **Honest limits, same as before.** Positive windows still come from only 5 CB events, and the
> held-out test rests on **one** event (~27 positive windows). Treat domain numbers as *directional*;
> the AI-Hub/in-domain negatives are what make precision meaningful. Exclusion (배제) is **not** this
> module's job — that's the metadata/SNA module (Phase 3). This head only sees verbal escalation.


In [1]:
# ---- run FIRST, before any torch/transformers import (avoids the multi-GPU NCCL error) ----
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single GPU; change to a free index if 0 is busy (nvidia-smi)
for k in ("RANK","LOCAL_RANK","WORLD_SIZE","MASTER_ADDR","MASTER_PORT"):
    os.environ.pop(k, None)

In [2]:
# On this server torch is already installed — do NOT reinstall it.
# If needed once: pip install transformers datasets scikit-learn pandas pyarrow accelerate
import json, zipfile, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix, classification_report, precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight
import torch, torch.nn as nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset
import transformers

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| visible GPUs:", torch.cuda.device_count())
if DEVICE == "cpu":
    print("⚠️  CPU only — set SMOKE_TEST=True first; a GPU runtime is strongly recommended.")

Device: cuda | visible GPUs: 1


In [3]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
PREP_DIR   = Path("./prepared")                          # from Phase 0-1
AIHUB_DIRS = [Path.home()/"Downloads/aihub/snsdata",     # multi-turn dialogue zips
              Path.home()/"Downloads/aihub/topicdata"]   # same schema (주제별) — included if present
MODEL_NAME = "klue/roberta-base"
OUT_DIR    = Path("./models/phase2_context")

WINDOW           = 6        # messages per window
WITHU_STRIDE     = 1        # dense (positives are scarce)
AIHUB_STRIDE     = 3        # sparser (negatives are plentiful)
AIHUB_CAP_WINDOWS = 10000   # cap AI-Hub negatives so they don't swamp the signal
AIHUB_CAP_CONVOS  = 4000

EPOCHS, BS, LR = 3, 16, 2e-5
SMOKE_TEST = False
SEED = 42
OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

## 1. Window construction (relative-speaker rendering)

In [4]:
def render_window(turns):
    """turns = [(speaker, text), ...] -> speaker-tagged string with turn separators.
    Speakers are remapped to relative A/B/C per window so the model learns turn structure,
    not specific (pseudonymous) names."""
    seen, letters, parts = {}, "ABCDEFGH", []
    for spk, txt in turns:
        spk = str(spk)
        if spk not in seen: seen[spk] = letters[len(seen) % len(letters)]
        parts.append(f"{seen[spk]}: {txt}")
    return " [TURN] ".join(parts)

def build_withu_windows(dom, W=WINDOW, stride=WITHU_STRIDE):
    ev = dom[dom.event_id.notna()].copy()
    cb_events = set(ev.groupby("event_id").cb_binary.max().pipe(lambda s: s[s == 1]).index)
    rows = []
    for eid, g in ev.groupby("event_id", sort=False):
        g = g.reset_index(drop=True)
        turns = list(zip(g.speaker.astype(str), g.text.astype(str)))
        labs  = g.cb_binary.fillna(0).astype(int).tolist()
        n = len(g)
        for i in range(0, max(1, n - W + 1), stride):
            idx = range(i, min(i + W, n))
            rows.append({"text": render_window([turns[j] for j in idx]),
                         "labels": int(any(labs[j] == 1 for j in idx)),
                         "event_id": eid, "is_cb_event": eid in cb_events})
    return pd.DataFrame(rows)

def build_aihub_windows(dirs, W=WINDOW, stride=AIHUB_STRIDE,
                        cap_windows=AIHUB_CAP_WINDOWS, cap_convos=AIHUB_CAP_CONVOS,
                        prefixes=("TL_", "VL_"), min_len=2):
    """Stream multi-turn dialogues straight from the AI-Hub label zips (no disk extraction).
    Every window is a CLEAN negative (label 0)."""
    dirs = [Path(d) for d in dirs if Path(d).exists()]
    zips = [z for d in dirs for z in sorted(d.glob("*.zip")) if z.name.startswith(prefixes)]
    print(f"AI-Hub: {len(zips)} label zips across {len(dirs)} dir(s)")
    rows, n_convo = [], 0
    for z in zips:
        if len(rows) >= cap_windows or n_convo >= cap_convos: break
        with zipfile.ZipFile(z) as zf:
            for name in zf.namelist():
                if not name.endswith(".json"): continue
                if len(rows) >= cap_windows or n_convo >= cap_convos: break
                try: d = json.loads(zf.read(name))
                except Exception: continue
                turns = [(t.get("speaker"), (t.get("text") or "").strip())
                         for t in d.get("utterances", [])]
                turns = [t for t in turns if len(t[1]) >= min_len]
                if len(turns) < 2: continue
                n_convo += 1
                for i in range(0, max(1, len(turns) - W + 1), stride):
                    rows.append({"text": render_window(turns[i:i+W]), "labels": 0,
                                 "event_id": None, "is_cb_event": False})
                    if len(rows) >= cap_windows: break
    print(f"  built {len(rows)} AI-Hub windows from {n_convo} conversations")
    return pd.DataFrame(rows)

## 2. Leak-free split — positives by event, negatives seeded everywhere

CB-event windows are held out **whole** (one event → val, one → test). WithU non-CB windows and
AI-Hub windows are split randomly so **both classes and in-domain negatives appear in every split**.


In [5]:
def split_windows(withu_win, aihub_win, n_test_cb=1, n_val_cb=1,
                  val_frac=0.15, test_frac=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    w = withu_win.copy(); w["split"] = "train"
    cb = list(sorted(w[w.is_cb_event].event_id.unique())); rng.shuffle(cb)
    if n_test_cb + n_val_cb >= len(cb):
        import warnings; warnings.warn(
            f"Only {len(cb)} CB events; holding out {n_test_cb+n_val_cb}. Val/test ≈ 1 event — directional only.")
    test_ev = set(cb[:n_test_cb]); val_ev = set(cb[n_test_cb:n_test_cb+n_val_cb])
    w.loc[w.event_id.isin(test_ev), "split"] = "test"
    w.loc[w.event_id.isin(val_ev),  "split"] = "val"
    non = w[~w.is_cb_event].index.to_numpy().copy(); rng.shuffle(non); n = len(non)
    w.loc[non[:int(n*test_frac)], "split"] = "test"
    w.loc[non[int(n*test_frac):int(n*(test_frac+val_frac))], "split"] = "val"

    a = aihub_win.copy(); a["split"] = "train"
    ai = a.index.to_numpy().copy(); rng.shuffle(ai); m = len(ai)
    a.loc[ai[:int(m*test_frac)], "split"] = "test"
    a.loc[ai[int(m*test_frac):int(m*(test_frac+val_frac))], "split"] = "val"

    allw = pd.concat([w, a], ignore_index=True)
    return allw, {"test_cb_event": sorted(test_ev), "val_cb_event": sorted(val_ev)}

## 3. Build the dataset

In [6]:
dom = pd.read_parquet(PREP_DIR / "withu_domain.parquet")
withu_win = build_withu_windows(dom)
aihub_win = build_aihub_windows(AIHUB_DIRS)

if len(aihub_win) == 0:
    print("⚠️  No AI-Hub windows found — check AIHUB_DIRS / that the zips start with TL_ or VL_. "
          "Training will still run on WithU-only negatives, but with weaker generalization.")

data, meta = split_windows(withu_win, aihub_win)
if SMOKE_TEST:
    data = data.groupby("split", group_keys=False).apply(lambda g: g.sample(min(len(g), 400), random_state=SEED))

print("split events:", meta)
print(data.groupby(["split", "labels"]).size().unstack(fill_value=0))

def to_ds(split):
    df = data[data.split == split][["text", "labels"]].reset_index(drop=True)
    return Dataset.from_pandas(df, preserve_index=False)

AI-Hub: 31 label zips across 2 dir(s)
  built 10000 AI-Hub windows from 2400 conversations
split events: {'test_cb_event': ['WCB005'], 'val_cb_event': ['WCB003']}
labels     0    1
split            
test    1531   27
train   7146  452
val     1531   33


## 4. Tokenize, class weights, weighted Trainer (same pattern that ran in Phase 1)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(ds): return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=256), batched=True)
train_ds, val_ds, test_ds = tok(to_ds("train")), tok(to_ds("val")), tok(to_ds("test"))

y_train = np.array(train_ds["labels"])
cw = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train)
class_weights = torch.tensor(cw, dtype=torch.float)
print("class weights [neg,pos]:", np.round(cw,3))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f}

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw); self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# processing_class on new transformers, tokenizer on old
_tok_kw = ({"processing_class": tokenizer} if transformers.__version__ >= "4.46"
           else {"tokenizer": tokenizer})

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Map:   0%|          | 0/7598 [00:00<?, ? examples/s]

Map:   0%|          | 0/1564 [00:00<?, ? examples/s]

Map:   0%|          | 0/1558 [00:00<?, ? examples/s]

class weights [neg,pos]: [0.532 8.405]


## 5. Train KLUE-RoBERTa window classifier

In [8]:
def make_args(output_dir, epochs, bs, lr):
    common = dict(output_dir=str(output_dir), num_train_epochs=epochs,
                  per_device_train_batch_size=bs, per_device_eval_batch_size=64,
                  learning_rate=lr, weight_decay=0.01, warmup_ratio=0.1,
                  logging_steps=50, save_total_limit=1, seed=SEED,
                  load_best_model_at_end=True, metric_for_best_model="f1", report_to="none")
    try:    return TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **common)
    except TypeError: return TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **common)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer = WeightedTrainer(
    class_weights=class_weights, model=model,
    args=make_args(OUT_DIR, EPOCHS if not SMOKE_TEST else 1, BS, LR),
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics, **_tok_kw)
trainer.train()
trainer.save_model(str(OUT_DIR)); tokenizer.save_pretrained(str(OUT_DIR))
print("saved ->", OUT_DIR)

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.041822,0.006482,0.987852,0.634615,1.000000,0.776471
2,0.063647,0.035299,0.993606,0.848485,0.848485,0.848485
3,0.000609,0.113529,0.987212,0.842105,0.484848,0.615385


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved -> models/phase2_context


## 6. Evaluate (directional on the held-out CB event) + pick threshold on val

In [9]:
def probs_labels(ds):
    out = trainer.predict(ds)
    p = torch.softmax(torch.tensor(out.predictions), dim=-1)[:,1].numpy()
    return p, out.label_ids

pv, yv = probs_labels(val_ds)
prec, rec, thr = precision_recall_curve(yv, pv)
f1s = 2*prec*rec/(prec+rec+1e-9)
best_t = float(thr[max(0, f1s[:-1].argmax())]) if len(thr) else 0.5
print(f"F1-optimal context threshold (val): {best_t:.3f}\n")

pt, yt = probs_labels(test_ds)
print("===== CONTEXT window test (directional — one held-out CB event) =====")
print(classification_report(yt, (pt>=best_t).astype(int), target_names=["normal","harmful-context"],
                            digits=4, zero_division=0))
print("confusion [rows=true, cols=pred]:\n", confusion_matrix(yt, (pt>=best_t).astype(int)))

(OUT_DIR/"context_meta.json").write_text(json.dumps(
    {"threshold": best_t, "window": WINDOW, "model": MODEL_NAME,
     "note": "positives from 5 CB events; test is single-event, directional"},
    ensure_ascii=False, indent=2))

F1-optimal context threshold (val): 0.026



===== CONTEXT window test (directional — one held-out CB event) =====
                 precision    recall  f1-score   support

         normal     0.9980    0.9941    0.9961      1531
harmful-context     0.7273    0.8889    0.8000        27

       accuracy                         0.9923      1558
      macro avg     0.8627    0.9415    0.8980      1558
   weighted avg     0.9933    0.9923    0.9927      1558

confusion [rows=true, cols=pred]:
 [[1522    9]
 [   3   24]]


160

## 7. Inference helper — score a live window

In [10]:
@torch.no_grad()
def score_context(recent_messages, model_dir=OUT_DIR, threshold=None):
    """recent_messages: list of (speaker, text) OR list of str (last ~6). Returns P(harmful context)."""
    tok_ = AutoTokenizer.from_pretrained(str(model_dir))
    mdl = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to(DEVICE).eval()
    turns = [m if isinstance(m, (tuple, list)) else ("?", m) for m in recent_messages]
    enc = tok_(render_window(turns), return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    p = float(torch.softmax(mdl(**enc).logits, dim=-1)[0, 1])
    thr = threshold if threshold is not None else json.loads((Path(model_dir)/"context_meta.json").read_text())["threshold"]
    return {"context_score": p, "harmful_context": bool(p >= thr)}

print(score_context([("A","야 너 왜 자꾸 그래"),("B","ㅋㅋㅋ 쟤 또 시작"),("A","진짜 꺼지라고"),
                     ("C","ㅋㅋㅋㅋ"),("A","아무도 너 안좋아해"),("B","맞아 나가")]))
print(score_context([("A","오늘 급식 뭐야?"),("B","돈까스래"),("A","오 좋아"),("B","ㅋㅋ 기대된다")]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'context_score': 0.0004016067832708359, 'harmful_context': False}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'context_score': 0.0002928374451585114, 'harmful_context': False}


In [11]:
import numpy as np
from sklearn.metrics import roc_auc_score

pt, yt = probs_labels(test_ds)
print("test positives: mean p = %.4f (n=%d)" % (pt[yt==1].mean(), (yt==1).sum()))
print("test negatives: mean p = %.4f (n=%d)" % (pt[yt==0].mean(), (yt==0).sum()))
print("threshold best_t = %.4f" % best_t)
print("ROC-AUC on test =", roc_auc_score(yt, pt) if len(set(yt))>1 else "n/a")

# score a few REAL held-out windows through the model (isolates OOD vs broken)
@torch.no_grad()
def score_rendered(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    return float(torch.softmax(trainer.model(**enc).logits, dim=-1)[0,1])

print("\nREAL held-out POSITIVE windows:")
for t in data[(data.split=="test") & (data.labels==1)].text.head(3):
    print("  %.4f  %s" % (score_rendered(t), t[:80]))
print("REAL held-out NEGATIVE windows:")
for t in data[(data.split=="test") & (data.labels==0)].text.head(3):
    print("  %.4f  %s" % (score_rendered(t), t[:80]))

test positives: mean p = 0.8886 (n=27)
test negatives: mean p = 0.0060 (n=1531)
threshold best_t = 0.0255
ROC-AUC on test = 0.9986936642717178

REAL held-out POSITIVE windows:
  0.9988  A: 아.. [TURN] A: 우울하다 [TURN] B: ᄏᄏᄏᄏᄏᄏᄏᄏᄏᄏ들킴? [TURN] C: 얼공도해 [TURN] A: 못생겨서하기싫음진
  0.9987  A: 우울하다 [TURN] B: ᄏᄏᄏᄏᄏᄏᄏᄏᄏᄏ들킴? [TURN] C: 얼공도해 [TURN] A: 못생겨서하기싫음진지하게 [TURN] D: 
  0.9962  A: ᄏᄏᄏᄏᄏᄏᄏᄏᄏᄏ들킴? [TURN] B: 얼공도해 [TURN] C: 못생겨서하기싫음진지하게 [TURN] D: 이미봤는데 [TURN] A:
REAL held-out NEGATIVE windows:
  0.9986  A: 소신발언) [TURN] A: 할거업으 [TURN] B: 소 신발 언 [TURN] A: 헉 [TURN] B: 사람들이지쳤잖아힘들잖아그만해야하
  0.0002  A: ᄏᄏᄏᄏᄏᄏ [TURN] B: 어.. 저기 채원아 갑자기 이런말해서 당혹스러울지도 모르는데 오늘이 아니면 언제해야할지 용기가 안나 나 너 
  0.0003  A: 모든 유미르의 백성들에게 고한다.
내 이름은 에렌 예거
시조의 거인의 힘을 빌려 모든 유미르의 백성들에게 말을 거는 중이다.
파라디섬의 있


## 8. What you have, and what's next

**Saved:** `./models/phase2_context/` — window-level context scorer (`score_context`) + `context_meta.json`.

**Two modules of the ensemble are now in place:** Phase 1 emits `p_toxic` per message, Phase 2 emits
`context_score` per window. Next:
- **Phase 3 — behavior/exclusion (배제)**: metadata + social-network analysis over live logs (no text
  model; this is where WEX events + the messenger's read/response/leave signals come in).
- **Phase 4 — bystander (방어/동조/방관)**: LLM few-shot bootstrap from your labeling rules; 방관 by log rules.
- **Ensemble + API**: combine `p_toxic` + `context_score` (+ later modules) → `cb_score`, apply the
  .75/.85 gates, serve behind FastAPI.

**Repeat the caveat in any writeup:** context positives come from 5 events and the test is one event —
directional; the in-domain + AI-Hub negatives are what make the precision figure trustworthy.
